# Laboratorio 1 

## Integrantes

Santiago Tenjo 202113965

# Importancion de librerias


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Exploración de los datos

In [18]:
datos_lab = pd.read_csv('Datos_Lab_1.csv')

## Información general y tipo de datos

In [19]:
# Información general del DataFrame
datos_lab.info()

<class 'pandas.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              2504 non-null   str    
 1   presion_media      2501 non-null   float64
 2   presion_min        2502 non-null   float64
 3   presion_max        2512 non-null   float64
 4   presion_desv       2496 non-null   float64
 5   humedad_media      2502 non-null   float64
 6   humedad_min        2496 non-null   float64
 7   humedad_max        2490 non-null   float64
 8   humedad_desv       2515 non-null   float64
 9   viento_media       2490 non-null   float64
 10  viento_min         2485 non-null   float64
 11  viento_max         2495 non-null   float64
 12  viento_desv        2504 non-null   float64
 13  rafaga_media       2498 non-null   float64
 14  rafaga_min         2504 non-null   float64
 15  rafaga_max         2503 non-null   float64
 16  rafaga_desv        2497 non-null   

In [20]:
# Verificar si hay valores nulos en el DataFrame
print(datos_lab.isnull().sum()[datos_lab.isnull().sum() > 0])

fecha                72
presion_media        75
presion_min          74
presion_max          64
presion_desv         80
humedad_media        74
humedad_min          80
humedad_max          86
humedad_desv         61
viento_media         86
viento_min           91
viento_max           81
viento_desv          72
rafaga_media         78
rafaga_min           72
rafaga_max           73
rafaga_desv          79
viento_norte         80
viento_este          64
direccion_viento     79
registros_del_dia    77
anio                 86
dia_del_anio         68
estacion_anio        88
mes                  88
sector_viento        71
temp_max_manana      95
dtype: int64


In [21]:
# Verificar valores duplicados en el DataFrame
duplicados_totales = datos_lab.duplicated().sum()
duplicados_fecha = datos_lab.duplicated(subset=['fecha']).sum()

print(f"Filas totalmente duplicadas: {duplicados_totales}")
print(f"Fechas duplicadas (Principio de Unicidad): {duplicados_fecha}")

Filas totalmente duplicadas: 4
Fechas duplicadas (Principio de Unicidad): 89


In [23]:
# Identificar estadisticas descriptivas

des = datos_lab[['humedad_min', 'presion_media', 'viento_media', 'temp_max_manana']].describe()
print(des)

       humedad_min  presion_media  viento_media  temp_max_manana
count  2496.000000    2501.000000   2490.000000      2481.000000
mean     45.094067    1010.494271      3.814103        13.863309
std      30.196163     435.334494      3.307402         9.296199
min       0.237700     948.996000      0.000000       -15.230000
25%       1.011790     984.287800      1.655350         7.310000
50%      49.670000     989.452800      2.463400        14.310000
75%      67.665000     994.469000      4.880505        20.460000
max     103.311405    9939.918000     24.753600        63.050000


# Carga de datos

In [25]:
datos_lab =  pd.read_csv('Datos_Lab_1.csv')

In [26]:
data =  datos_lab.copy()

# Limpieza de los datos

In [ ]:

# Convertimos 'fecha' a tipo datetime para poder ordenar

data['fecha'] = pd.to_datetime(data['fecha'], format='%Y-%m-%d')

# Ordenar cronológicamente (vital para modelos basados en tiempo)
data = data.sort_values('fecha').reset_index(drop=True)



In [28]:

# Correcion icnossitencias variables categoricas

data['mes'] = data['mes'].astype(str).str.strip().str.capitalize()
data['estacion_anio'] = data['estacion_anio'].astype(str).str.strip().str.lower()
data['sector_viento'] = data['sector_viento'].astype(str).str.strip().str.upper()

In [30]:

# Tratamiento de duplicados y unicidad
data.drop_duplicates()
data.drop_duplicates(subset=['fecha'], keep='first')

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.77860,...,2.255577,0.8488,-0.3618,-0.0223,183.5308,143.0,invierno,January,S,-2.120
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.41950,...,6.130000,1.2544,0.4267,0.3689,40.8436,144.0,invierno,July,NE,-0.820
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.25090,...,4.880000,0.9330,-0.6993,-0.5268,216.9916,144.0,invierno,January,SO,-0.630
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.72040,...,4.430375,1.1835,-1.1268,-1.0413,222.7419,144.0,invierno,January,SO,-1.440
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.80030,...,10.880000,NaN,2.6275,0.2874,6.2418,NaN,east,January,N,-10.880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2552,2015-12-28,1003.5725,1002.35,1004.96,0.7294,87.109700,72.100000,97.7,7.5170,3.02724,...,3.840000,-29.5050,-0.6644,-0.0853,187.3193,144.0,verano,March,S,9.160
2553,2015-12-29,1002.1544,1000.83,1005.33,1.2520,95.091000,79.400000,99.5,5.3604,0.86610,...,3.560000,0.7173,-0.5544,-0.0871,188.9267,144.0,invierno,December,S,5.650
2554,2015-12-30,1002.9883,997.51,1005.73,2.5524,84.473700,NaN,99.4,12.9567,2.12740,...,0.640000,2.0483,-2.0540,-0.1130,183.1497,144.0,invierno,Diciembre,SOUTH,2.920
2555,2015-12-31,996.9224,995.01,999.20,1.4003,0.726064,0.495600,97.4,17.6803,3.11310,...,9.090000,1.9769,-3.0401,0.1673,176.8493,144.0,invierno,NaN,S,2.720


In [33]:
#  Uniformas escalas a porcentual
if 'humedad_min' in data.columns:
        data.loc[data['humedad_min'] <= 1, 'humedad_min'] *= 100
if 'humedad_max' in data.columns:
        data.loc[data['humedad_max'] <= 1, 'humedad_max'] *= 100

In [ ]:

# Eliminar columnas redundantes para evitar multicolinealidad
data = data.drop(columns=['anio', 'dia_del_anio'])

In [ ]:
# Gestion de variables objetivo 
# Si quedan nulos en la variable objetivo se deben eliminar

if 'temp_max_manana' in data.columns:
        data = data.dropna(subset=['temp_max_manana'])

# Construcción de un modelo de regresión lineal: